# Reproducing Didion-Gency et al. (2024) from Dryad dataset 4j0zpc8hz

Paper. Didion-Gency M, Vitasse Y, Buchmann N, Gessler A, Gisler J, Schaub M, Grossiord C (2024).
Chronic warming and dry soils limit carbon uptake and growth despite a longer growing season in beech
and oak. Plant Physiology 194, [doi:10.1093/plphys/kiad565](https://doi.org/10.1093/plphys/kiad565).
The claims file was written from the version of record in Europe PMC (PMC10828195) with its
supplement (Supplemental Figures S1 to S7, Tables S1 to S6, Methods S1 to S4); no figure was seen.

Dataset. [doi:10.5061/dryad.4j0zpc8hz](https://doi.org/10.5061/dryad.4j0zpc8hz), version id 260590 in
Dryad's API (version number 3, published 2023-10-17). The paper cites the dataset's DOI, which resolves
to the latest version, and no version. Dryad lists only this one version, so it is pinned. Files read,
with the SHA-256 that Dryad's file list gives and that the first cell checks:

| File | SHA-256 |
|---|---|
| Dryad_Data_PlantPhy_MargauxDidionGency.xlsx | b18a8e25527e2f5ecc80a9d520913b38eafb4382fe7e6c5e8ee7232e243300a3 |
| Metadata.csv | eaffc99d0e0a23c7e3bab236d4ecd951c92733cacbf9d7c8a57447a9b5ad0766 |

The dataset's README.md (the same text as the workbook's README sheet) is the third file of the
version; the analysis does not use it. Dryad needs a token to download, so the files are staged
before the notebook runs and read in place (docs/archives.md, "Dryad").

What the dataset holds. One table, one row per tree and measurement campaign: 96 trees (beech and
oak, monocultures and mixtures, 6 per treatment, species and mixture, in 16 chambers), 3 campaigns a
year in 2019 to 2021, with the yearly phenology, the treatment's mean air temperature per season, GDD
and chilling days per tree, the leaf-level traits per campaign (Asat, sugar, starch, predawn and midday
water potential) and the yearly height, diameter and AGWB increments. It has no soil moisture, no
chamber climate records and no initial tree sizes.

Outside inputs. None. AGWB increments are in the dataset, so the coefficients of Annighofer et al.
(2016) are not needed.

| Claim | Tested |
|---|---|
| fig1ab-bud-swelling-advance | yes |
| fig1cd-development-duration | yes |
| fig1gh-senescence-duration | yes |
| fig1ij-growing-season | yes |
| fig1-oak-steeper-advance | yes |
| fig1ef-onset-senescence-no-temperature | yes |
| figs1-phenology-no-moisture | no, data missing |
| fig2-gdd-chilling | yes |
| disc-max-chilling-days | yes |
| fig3-psi-warming-2020 | yes |
| fig3-psi-mr-oak-2020 | yes |
| fig3-psi-wmr-2021 | yes |
| disc-psi-minimum | yes |
| fig4-beech-warming-none | yes |
| fig4-oak-warming-growth-2021 | yes |
| disc-oak-warming-no-carbon-change | yes |
| fig4-beech-mr | yes |
| fig4-oak-mr | yes |
| fig4-beech-wmr | yes |
| fig4-oak-wmr | yes |
| fig4-wmr-same-as-mr | yes |
| disc-nsc-unchanged-drier-soil | yes |
| disc-beech-asat-more-impaired | yes |
| tables6-no-seasonal-variation | yes |
| tables2-phenology-no-interaction | yes |
| tables3-beech-warming-no-interaction | yes |
| figs2-beech-asat-selection-warming | yes |
| tables3-oak-warming-diameter | yes |
| fig5-oak-agwb-nbe-warming-2020 | yes |
| tables3-beech-mr-diameter | yes |
| fig5-beech-sugar-nbe-mr | yes |
| oak-mr-no-interaction | yes |
| tables3-beech-wmr-height | yes |
| tables3-oak-wmr-diameter | yes |
| fig5-oak-asat-nbe-wmr-2021 | yes |
| methods-treatment-magnitudes | no, data missing |
| fig6-soil-moisture-mr-equals-wmr | no, data missing |
| tables5-annual-climate | no, data missing |
| tables4-initial-sizes-similar | no, data missing |

Readings. Where the claims file leaves a detail open it names readings (P1/P2, T1/T2, S1/S2/S3, L1/L2,
I1/I2, Y1/Y2, B1/B2, and for Table S6 y1/y2 and F1/F2). Every reading is run. A first run of this
attempt asked three questions (questions/dryad-4j0zpc8hz.yaml); a reader answered them from the paper
and the claims file was updated, and this version of the notebook follows the updated file. The paper
does not say how negative values were square-rooted, so negative starch, sugar and Asat values are read
as negative increments are, under S1 (untransformed), S2 (square root after adding the constant that
brings the minimum to zero) and a new reading S3 (negative values dropped, as R's sqrt gives NaN for
them). The interaction model takes the years of reading Y1 (bud swelling and development duration on
2020 and 2021, the other traits on 2019 to 2021, as the F and p pairs of Tables S2 and S3 imply), which
the criteria use, with Y2 (2020 and 2021 for all traits) as a variant. The Table S6 model is run on
2019 to 2021 (y1) and on 2020 and 2021 (y2), with sequential (F1) and marginal (F2) F tests, all of
which are readings of its criterion.

After the first verification round, three more questions were asked (questions/dryad-4j0zpc8hz.yaml):
over which family the mixture against monoculture contrasts are adjusted, run here as A1 (each
contrast alone) and A2 (single-step over the contrasts of the model); whether Eqs. 3 and 4 are to
be read so that the complementarity effect is twice the NBE and the selection effect is 0 under B1;
and whether growing season length is the deposited Growth_length or bud swelling to onset of
senescence. Two variants that are not readings of the claims file are also shown: the regressions of
reading P1 with a random intercept per treatment and year, and the Table S6 F values set against the
paper's.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import oaktrees as o

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
warnings.filterwarnings("ignore")
started = time.time()

print(f"Dryad dataset {o.DOI}, version id {o.VERSION_ID}; files read in place from {o.data_dir()}")
print(o.check_files().to_string(index=False))
workbook = o.read_workbook()
metadata = o.read_metadata()
raw = workbook["Total Data"]
d = o.load(raw)
ty = o.tree_year_table(d)          # one row per tree and year (leaf-level traits: yearly tree means, L1)
ca = o.campaign_table(d)           # one row per tree and campaign (L2)
print(f"loaded in {time.time() - started:.1f} s")

## Data check

Each check of the description against the values is stated under its output. The dataset has no time
series (no chamber climate records), so there is no clock to align; the only time variables are the year
and the campaign number.

In [ ]:
print("Sheets of the workbook:", {k: v.shape for k, v in workbook.items()})
print("\nColumns of 'Total Data':", list(raw.columns))
print("\nMetadata.csv entries:", list(metadata["Entry"].str.strip()))
print("Columns named in Metadata.csv but not in the sheet:", sorted(set(metadata["Entry"].str.strip()) - set(raw.columns)))
print("Columns of the sheet not in Metadata.csv:", sorted(set(raw.columns) - set(metadata["Entry"].str.strip())))
print("\nTreatment codes in the data:", raw["Treatment"].value_counts().sort_index().to_dict())
print("Metadata.csv says:", metadata.loc[metadata.Entry.str.strip() == "Treatment", "Explaination"].iloc[0])
print(pd.crosstab(raw["Treatment"], [raw["Warming"], raw["Soil moisture reduction"]]))
print("\nRows:", len(raw), "; exact duplicate rows:", int(raw.duplicated().sum()),
      "; trees with duplicates:", sorted(raw[raw.duplicated(keep=False)].Tree.unique().tolist()))
dup = d[d.duplicated(["Tree", "Year", "Campaign"], keep=False)]
print("Rows left with the same tree, year and campaign after dropping exact duplicates:")
print(dup[["Tree", "Year", "Campaign", "Photo", "Sugar", "Starch", "Predawn", "Midday"]].to_string(index=False))
print("\nOne chamber, treatment, species and mixture per tree:",
      bool((d.groupby("Tree")[["Chamber", "treat", "Species", "Mixture", "Compartment"]].nunique() == 1).all().all()))
trees = d.drop_duplicates("Tree")
print("\nTrees per treatment, mixture and species (paper: 6 each, 96 in all):")
print(pd.crosstab([trees.treat, trees.Mixture], trees.Species))
print("\nChambers per treatment (paper: 4):", trees.groupby("treat").Chamber.nunique().to_dict())
print("\nCampaigns per year:", d.groupby("Year").Campaign.unique().apply(sorted).to_dict())
print("Tree-campaign rows per year, species and mixture:")
print(pd.crosstab([ca.Year], [ca.Species, ca.Mixture]))

The metadata name the treatments C, W, MR and WMR; the data use C, D, H and HD. The Warming and Soil
moisture reduction columns map them one to one: H is warming (W), D is moisture reduction (MR) and HD
both (WMR). The analysis uses the paper's names, from those two columns. 35 rows repeat another row
exactly (trees 17, 35 and 45). They are counted once. Tree 144 also has two different rows for campaign 5
of 2020 (same Asat and water potentials, different sugar and starch), which are averaged. The design
matches the paper: 96 trees, 6 per treatment, species and mixture, 4 chambers per treatment, 3 campaigns
a year in 2019 to 2021.

In [ ]:
cols = ["Flush_S1", "Dev_time", "Senes_10", "Senes_time", "Growth_length", "Tspring", "Tfall", "Tannual", "GDD",
        "Chill_days", "Photo", "Sugar", "Starch", "Predawn", "Midday", "Height_growth", "Diameter_growth", "AGB_growth"]
units = metadata.set_index(metadata.Entry.str.strip())["Unit"]
rng = pd.DataFrame({"unit": units.reindex(cols), "n": ca[cols].count(), "min": ca[cols].min(), "median": ca[cols].median(),
                    "max": ca[cols].max(), "n_negative": (ca[cols] < 0).sum(), "n_zero": (ca[cols] == 0).sum()})
print("Ranges over tree-campaign rows (yearly traits repeat on the three rows of a year):")
print(rng.round(3).to_string())
print("\nYearly traits constant within a tree and year:",
      bool((d.groupby(["Tree", "Year"])[o.YEARLY].nunique() <= 1).all().all()))
clim = d.groupby(["Year", "treat"])[["Tspring", "Tfall", "Tannual"]].agg(["nunique", "first"])
print("\nTspring, Tfall and Tannual per year and treatment (number of distinct values, value):")
print(clim.round(3).to_string())
print("Distinct values per year and chamber:", int(d.groupby(["Year", "Chamber"])[["Tspring", "Tfall", "Tannual"]].nunique().max().max()))

Ranges and units are plausible for the phenology dates (days of year 64 to 291) and durations. The air
temperatures are one value per treatment and year, the same in the four chambers of a treatment: the
dataset gives no chamber climate, and the sensor height (50 cm or 2 m) is not stated. Starch (77 rows),
sugar (3) and Asat (5) have negative values, which a concentration or a light-saturated assimilation
rate cannot have. Height increments are negative in 30 rows and diameter increments in 90, which a
shrinking stem can give only within measurement error. Soil moisture, the chamber climate records
(daily air temperature, VPD, soil water content), the initial heights and diameters (Table S4) and the
absolute heights, diameters and AGWB are not in the dataset.

In [ ]:
y = ty[ty.Year >= 2020]
desc = metadata.assign(Entry=metadata.Entry.str.strip()).set_index("Entry")
readme = workbook["README.md"]
readme = readme.set_index(readme.columns[1])
print("How the dataset describes the derived variables (Metadata.csv; the workbook's README sheet):")
for col in ("Growth_length", "GDD", "Chill_days"):
    print(f"  {col} [{desc.loc[col, 'Unit']}]: {desc.loc[col, 'Explaination']}")
    print(f"  {'':{len(col)}}  README sheet: {readme.loc[col].iloc[3]}")
print("The Dryad README.md file holds the title, the citation and a contact; it describes no column.\n")
calc = y.Senes_10 - y.Flush_S1
print("Growing season length against the Methods' definition (onset of senescence minus bud swelling):")
print(f"  rows with both: {int(calc.notna().sum())}; equal: {int((calc == y.Growth_length).sum())}; "
      f"median of Growth_length - (Senes_10 - Flush_S1): {np.nanmedian(y.Growth_length - calc):.1f} days")
end90 = y.Senes_10 + y.Senes_time - y.Flush_S1
print(f"  against the end of senescence (10% date + duration) minus bud swelling: equal in {int((end90 == y.Growth_length).sum())}, "
      f"median difference {np.nanmedian(y.Growth_length - end90):.1f} days, range {np.nanmin(y.Growth_length - end90):.0f} to {np.nanmax(y.Growth_length - end90):.0f}")
print("\nOnset of senescence at its minimum (DOY 250 to 252):", int((y.Senes_10 <= 252).sum()), "of", int(y.Senes_10.notna().sum()), "tree-years")
print("\nDiameter increments (column unit 'cm'), monocultures and mixtures, tree-years 2020-21:")
print(y.Diameter_growth.describe().round(3).to_string())
print("Paper, Table S4: initial diameters of 5.2 to 6.1 mm.")
comp = d.drop_duplicates("Tree").groupby(["Chamber", "Compartment"]).apply(
    lambda g: "+".join(sorted(f"{s}-{m[:4]}" for s, m in zip(g.Species, g.Mixture))))
odd = comp[comp.str.contains("Mono") & comp.str.contains("Mixt")]
print("\nCompartments holding both a monoculture tree and a mixture tree:", len(odd), "of", len(comp))
print(odd.to_string())
agb0 = y[(y.AGB_growth == 0)]
print("\nAGWB increment exactly 0 in", len(agb0), "of", int(y.AGB_growth.notna().sum()), "tree-years 2020-21; their height increments:",
      agb0.Height_growth.describe().round(1).to_dict())

Growth_length never equals onset of senescence minus bud swelling, the Methods' definition of growing
season length; it is close to the end of senescence (10% date plus senescence duration) minus bud
swelling, but not equal to it either. The deposited Growth_length is used, as the claims file names the
dataset's growing season length; the Methods' definition is shown as a variant in fig1ij. The onset
of senescence sits at day 250 to 252 in many tree-years: the observations began in September, so these
values are censored at the first observation. Diameter increments of a median of about 0.5 "cm" a year
on stems of about 6 mm are implausible; the values are consistent with millimetres. Some compartments
hold a monoculture tree of one species and a mixture tree of the other, which the design (each
compartment a monoculture or a mixture) does not allow; the compartment labels are not used. AGWB
increments are 0 in many tree-years whose height increment is positive.

### The deposited data against the paper's Table S1

Table S1 gives F tests of the treatment model. This check refits it on 2019 to 2021 (as the table
does), on yearly tree means (L1) and on campaign values (L2), and gives both the sequential tests (the
claims file's reading) and the marginal ones, for traits whose values have no negatives, where the
square root is defined. It is a check of the data and of the model code, and decides no outcome.

In [ ]:
TABLE_S1 = {   # Supplemental Table S1, F values in the order W, MR, Y, W:MR, W:Y, MR:Y, W:MR:Y
    ("B", "Photo"): [2.44, 0.99, 25.86, 4.12, 5.22, 19.00, 2.75],
    ("O", "Photo"): [0.00, 2.70, 3.41, 0.17, 1.02, 0.66, 1.58],
    ("B", "AGB_growth"): [0.17, 0.04, 1.75, 0.09, 0.19, 2.83, 1.08],
    ("O", "AGB_growth"): [0.33, 0.93, 5.05, 0.89, 1.62, 7.27, 1.04],
    ("B", "Predawn"): [2.83, 0.10, 0.25, 0.00, 2.12, 0.12, 3.54],
    ("O", "Predawn"): [0.34, 0.07, 7.95, 2.34, 0.90, 1.70, 3.59],
    ("B", "Midday"): [0.96, 0.06, 6.50, 0.04, 2.10, 0.27, 0.97],
    ("O", "Midday"): [0.50, 0.04, 2.25, 0.00, 0.35, 0.95, 0.42],
}
rows = []
for (sp, trait), paper in TABLE_S1.items():
    sq = trait not in ("Predawn", "Midday")
    for L, data in (("L1", ty), ("L2", ca)):
        r = o.treatment_test(data, sp, trait, [2019, 2020, 2021], sq, "S1")
        for kind in ("anova", "anova_marginal"):
            F = r[kind].F.round(2).tolist()
            rows.append({"species": sp, "trait": trait, "L": L, "tests": "sequential" if kind == "anova" else "marginal",
                         "n": r["n"], "F": F, "max_abs_diff": float(np.max(np.abs(np.array(F) - paper)))})
link = pd.DataFrame(rows)
print(link.to_string(index=False))
best = link.loc[link.groupby(["species", "trait"]).max_abs_diff.idxmin()]
print("\nClosest version per trait:")
print(best[["species", "trait", "L", "tests", "max_abs_diff"]].to_string(index=False))

The marginal tests on the deposited data reproduce Table S1 for oak AGWB increment exactly (largest
difference in F 0.00, yearly tree means), for beech Asat closely (0.25, campaign values) and for oak
midday water potential within 0.02 (campaign values). The sequential tests, which the claims file
names, differ from the table by up to 68 in F. Other traits match less well. This is consistent with
the deposited data being the paper's data and with the paper's F tests being marginal; it decides no
outcome, and the question is asked (questions/dryad-4j0zpc8hz.yaml). The criteria below rest on the
contrasts, which do not depend on how the F tests are taken.

## Phenology regressions (Figure 1)

Phenology regression of the claims file: per species, monoculture trees, 2020 and 2021 pooled, a linear
regression of the trait on the season's air temperature. Reading P1 is each tree and year against the
temperature of its chamber; the dataset gives the temperature per treatment and year only, so P1 here
uses the treatment's value (the same in its four chambers). Reading P2 is the eight treatment and year
means. The sensor readings H1 and H2 cannot be told apart: the dataset gives one temperature and does not
say which sensor it comes from.

In [ ]:
import statsmodels.formula.api as smf

mono = ty[(ty.Mixture == "Monoculture") & ty.Year.isin([2020, 2021])]

def pheno(trait, temp):
    rows = []
    for sp in ("B", "O"):
        s = mono[mono.Species == sp].dropna(subset=[trait, temp])
        r1 = o.linear_fit(s[temp], s[trait])
        m = s.groupby(["treat", "Year"])[[trait, temp]].mean()
        r2 = o.linear_fit(m[temp], m[trait])
        rows += [{"species": sp, "reading": "P1", **r1}, {"species": sp, "reading": "P2", **r2}]
    return pd.DataFrame(rows)

def within(v, target, tol):
    return abs(v - target) <= tol + 1e-12

def judge_slopes(tab, targets):
    out = []
    for reading, g in tab.groupby("reading"):
        g = g.set_index("species")
        ok = all(within(g.loc[sp, "slope"], targets[sp][0], targets[sp][1]) and g.loc[sp, "p"] <= 0.05 for sp in "BO")
        out.append({"reading": reading, "slopes": g.slope.round(3).to_dict(), "p": g.p.map(lambda v: f"{v:.2g}").to_dict(),
                    "all_within_and_p<=0.05": ok})
    return pd.DataFrame(out)

def pheno_re(trait, temp):
    """Variant of P1 (not a reading of the claims file): each tree-year, with a random intercept per treatment
    and year, as the temperature is one value per treatment and year (REML, Wald z of the slope)."""
    rows = []
    for sp in ("B", "O"):
        s = mono[mono.Species == sp].dropna(subset=[trait, temp]).assign(g=lambda f: f.treat + f.Year.astype(str))
        f = smf.mixedlm(f"{trait} ~ {temp}", s, groups=s.g).fit(reml=True)
        rows.append({"species": sp, "reading": "P1 + random treatment-year", "slope": f.params[temp], "se": f.bse[temp],
                     "p": f.pvalues[temp], "n": len(s), "groups": s.g.nunique(), "converged": f.converged})
    return pd.DataFrame(rows)

PH, PHRE = {}, {}
for cid, trait, temp in [("fig1ab", "Flush_S1", "Tspring"), ("fig1cd", "Dev_time", "Tspring"), ("fig1ef", "Senes_10", "Tfall"),
                         ("fig1gh", "Senes_time", "Tfall"), ("fig1ij", "Growth_length", "Tannual")]:
    PH[cid] = pheno(trait, temp)
    PHRE[cid] = pheno_re(trait, temp)
print("Monoculture tree-years with data, 2020-21:", mono.groupby("Species")[["Flush_S1", "Dev_time", "Senes_10", "Senes_time", "Growth_length"]].count().to_dict("index"))
print("Temperature ranges (deg C):", {t: (round(mono[t].min(), 2), round(mono[t].max(), 2)) for t in ("Tspring", "Tfall", "Tannual")})

### fig1ab-bud-swelling-advance

> Warmer spring air advanced bud swelling in both species, by 1.1 days per degree in beech and 2.3 days per degree in oak, across 2020 and 2021 and the four treatments.

In [ ]:
print(PH["fig1ab"].round(4).to_string(index=False))
print(judge_slopes(PH["fig1ab"], {"B": (-1.1, 0.11), "O": (-2.3, 0.23)}).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, sp in zip(axes, "BO"):
    s = mono[mono.Species == sp]
    ax.scatter(s.Tspring, s.Flush_S1, s=8)
    ax.set_title(sp); ax.set_xlabel("Tspring (C)"); ax.set_ylabel("bud swelling (DOY)")
plt.tight_layout(); plt.show()
print("Plotted ranges: Tspring", round(mono.Tspring.min(), 2), "to", round(mono.Tspring.max(), 2), "; DOY", mono.Flush_S1.min(), "to", mono.Flush_S1.max())
print("\nVariant, P1 with a random intercept per treatment and year:")
print(PHRE["fig1ab"].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** Bud swelling advances by 1.08 days per degree
in beech (P1 p = 0.00011, P2 p = 0.033) and 2.30 in oak (P1 p = 1.2e-08, P2 p = 0.010), both within
tolerance under both readings. The slopes are the same under P1 and P2 because every treatment and
year has six trees; P1 uses the treatment's temperature, as the dataset has no chamber value. With a random intercept
per treatment and year added to P1 (a variant), p = 0.0060 (beech) and 0.0002 (oak).

### fig1cd-development-duration

> Warmer spring air shortened leaf development duration by 0.7 days per degree in both species.

In [ ]:
print(PH["fig1cd"].round(4).to_string(index=False))
print(judge_slopes(PH["fig1cd"], {"B": (-0.7, 0.07), "O": (-0.7, 0.07)}).to_string(index=False))
print("\nVariant, P1 with a random intercept per treatment and year:")
print(PHRE["fig1cd"].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** Development duration shortens by 0.65 days
per degree in beech and 0.66 in oak, within 0.7 +- 0.07. Under P1 both regressions have P <= 0.05
(0.0010 and 0.0039); under P2 the oak regression does not (p = 0.16). The criterion needs one
reading. P1 treats the 48 tree-years as independent although the temperature is one value per
treatment and year; with a random intercept per treatment and year (a variant, not a reading of the
claims file), oak has p = 0.11 and beech 0.012, so the oak result rests on P1 as the claims file
gives it.

### fig1gh-senescence-duration

> Warmer fall air extended leaf senescence duration by 3.4 days per degree in beech and 3.1 days per degree in oak.

In [ ]:
print(PH["fig1gh"].round(4).to_string(index=False))
print(judge_slopes(PH["fig1gh"], {"B": (3.4, 0.34), "O": (3.1, 0.31)}).to_string(index=False))
print("\nVariant, P1 with a random intercept per treatment and year:")
print(PHRE["fig1gh"].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** Senescence duration lengthens by 3.40 days per
degree in beech and 3.08 in oak, within tolerance, with P <= 0.0056 under both readings, and under the variant with a random intercept per treatment
and year (p < 0.0001 in both species).

### fig1ij-growing-season

> Warmer annual air lengthened the growing season by 4.5 days per degree in beech and 5.7 days per degree in oak.

In [ ]:
print(PH["fig1ij"].round(4).to_string(index=False))
print(judge_slopes(PH["fig1ij"], {"B": (4.5, 0.45), "O": (5.7, 0.57)}).to_string(index=False))
alt = mono.assign(GSL_methods=mono.Senes_10 - mono.Flush_S1)
rows = []
for sp in "BO":
    s = alt[alt.Species == sp].dropna(subset=["GSL_methods"])
    m = s.groupby(["treat", "Year"])[["GSL_methods", "Tannual"]].mean()
    rows += [{"species": sp, "reading": "P1", **o.linear_fit(s.Tannual, s.GSL_methods)},
             {"species": sp, "reading": "P2", **o.linear_fit(m.Tannual, m.GSL_methods)}]
print("\nVariant, growing season length as the Methods define it (Senes_10 - Flush_S1):")
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nVariant, P1 with a random intercept per treatment and year:")
print(PHRE["fig1ij"].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** Growing season length increases by 4.46 days per
degree in beech and 5.65 in oak, within tolerance, with P <= 0.035 under both readings. The deposited
Growth_length is not onset of senescence minus bud swelling, as the Methods define it (data check);
with the Methods' definition, which the criterion repeats (bud swelling to onset of senescence), the
slopes are 1.09 (beech, P1 p = 0.085) and 2.16 (oak, P1 p = 0.0014), and the claim would not be
reproduced. Neither the workbook nor Metadata.csv defines Growth_length (data check). The deposited
column gives the paper's slopes within 0.04, so it is taken as the paper's variable; a question asks
which is to be tested. With a random intercept per treatment and year (a variant), p = 0.0067 (beech)
and < 0.0001 (oak).

### fig1-oak-steeper-advance

> The temperature-driven advance was larger in oak than in beech (P = 0.004, least squares analysis).

In [ ]:
rows = []
for reading, trait, temp in [("a", "Flush_S1", "Tspring"), ("b", "Growth_length", "Tannual")]:
    s = mono.dropna(subset=[trait])[["Species", "treat", "Year", trait, temp]]
    for unit, data in (("P1", s), ("P2", s.groupby(["Species", "treat", "Year"], as_index=False)[[trait, temp]].mean())):
        fit = smf.ols(f"{trait} ~ {temp} * C(Species)", data).fit()
        term = f"{temp}:C(Species)[T.O]"
        slope_b = fit.params[temp]
        slope_o = slope_b + fit.params[term]
        rows.append({"reading": reading, "unit": unit, "slope_beech": slope_b, "slope_oak": slope_o,
                     "interaction_p": fit.pvalues[term], "n": int(fit.nobs),
                     "oak_larger_and_p<0.05": bool(abs(slope_o) > abs(slope_b) and fit.pvalues[term] < 0.05)})
steep = pd.DataFrame(rows)
print(steep.round(4).to_string(index=False))
rows = []
for reading, trait, temp in [("a", "Flush_S1", "Tspring"), ("b", "Growth_length", "Tannual")]:
    s = mono.dropna(subset=[trait]).assign(g=lambda f: f.Species + f.treat + f.Year.astype(str))
    fit = smf.mixedlm(f"{trait} ~ {temp} * C(Species)", s, groups=s.g).fit(reml=True)
    term = f"{temp}:C(Species)[T.O]"
    rows.append({"reading": reading, "unit": "P1 + random species-treatment-year", "slope_beech": fit.params[temp],
                 "slope_oak": fit.params[temp] + fit.params[term], "interaction_p": fit.pvalues[term], "n": len(s),
                 "groups": s.g.nunique(), "converged": fit.converged})
print("\nVariant, P1 with a random intercept per species, treatment and year (Wald z):")
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** Under reading (a), bud swelling on Tspring with
each tree and year (P1), the oak slope (-2.30) is steeper than the beech slope (-1.08), with species by
temperature interaction p = 0.0046 against the paper's 0.004. On the 16 treatment means (P2) p = 0.12,
and for growing season length (reading b) p = 0.36 (P1) and 0.57 (P2). With a random intercept per
species, treatment and year added to P1 (a variant, not a reading of the claims file), reading (a)
gives p = 0.097, so the result rests on P1 as the claims file gives it.

### fig1ef-onset-senescence-no-temperature

> The onset of senescence was not related to fall air temperature in either species.

In [ ]:
print(PH["fig1ef"].round(4).to_string(index=False))
print("Any regression with P <= 0.05:", bool((PH["fig1ef"].p <= 0.05).any()))
print("\nVariant, P1 with a random intercept per treatment and year:")
print(PHRE["fig1ef"].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** No regression of onset of
senescence on Tfall has P <= 0.05 (p = 0.44 to 0.80), with slopes of 0.24 (beech) and -0.45 (oak)
days per degree. The variant with a random intercept per treatment and year gives p = 0.79 and 0.45.

### fig2-gdd-chilling

> The more chilling days trees received, the fewer growing degree days they needed to reach bud swelling, in both species, following a negative exponential.

The dataset gives GDD and chilling days per tree and year, computed by the authors; the chamber
temperature records they come from are not in it. So the claims file's readings of how they are
computed ((a) or (b) for GDD, (c) or (d) for the end of the chilling count, sensor H1 or H2) cannot be
run one by one: the test uses the deposited values, whose description says chilling days are counted
to leaf flushing (reading d) and GDD accumulate "the daily mean temperature above a threshold of 5 C"
(a or b). Both forms of the negative exponential, (e) and (f), are fitted.

In [ ]:
rows = []
for sp in "BO":
    s = mono[mono.Species == sp].dropna(subset=["GDD", "Chill_days"])
    for form in ("e", "f"):
        r = o.nls_exponential(s.Chill_days, s.GDD, form)
        rows.append({"species": sp, **{k: r.get(k) for k in ("form", "n", "converged", "a", "b", "c", "rate", "p_rate", "r2", "decreasing")}})
gdd = pd.DataFrame(rows)
gdd["holds"] = gdd.converged & gdd.decreasing & (gdd.p_rate <= 0.05)
print(gdd.to_string(index=False))
print("Ranges: Chill_days", mono.Chill_days.min(), "to", mono.Chill_days.max(), "; GDD", round(mono.GDD.min(), 1), "to", round(mono.GDD.max(), 1))
print("Spearman correlation of GDD with chilling days:",
      {sp: round(mono[mono.Species == sp][["GDD", "Chill_days"]].corr("spearman").iloc[0, 1], 3) for sp in "BO"})

**Result: reproduced.** With the deposited GDD and chilling days, both forms
of the negative exponential decrease with chilling days in both species, with rate parameters of p =
4.4e-27 (beech, e), 2.9e-07 (beech, f), 7.4e-20 (oak, e) and 4.2e-07 (oak, f). The readings of how
GDD and chilling days are computed could not be run separately (the records are not in the
dataset).

### disc-max-chilling-days

> Even without warming, trees received at most 97 chilling days.

In [ ]:
unw = mono[mono.treat.isin(["C", "MR"])]
print("Maximum chilling days, C and MR monoculture trees 2020-21:", unw.Chill_days.max())
print(unw.groupby(["Year", "treat", "Species"]).Chill_days.agg(["min", "max", "count"]).to_string())
print("All trees and years in the dataset:", d.Chill_days.max())

**Result: reproduced.** The largest number of chilling days among C and
MR monoculture trees in 2020 and 2021 is 97 (in 2021), as is the largest in the whole dataset.

## Treatment tests (Figures 3 and 4)

Treatment test of the claims file: per species, monoculture trees, lme(trait ~ (W + MR + year)^3,
random = ~1|Chamber), fitted by REML; Tukey-type contrasts among the four treatments within each year,
with single-step adjusted p over the six pairs of a year. Square root for starch, sugar, Asat and the
increments, none for the water potentials. Readings: T1 (2019 to 2021) and T2 (2020 and 2021); where
some values to be square-rooted are negative (starch, sugar and Asat as well as the increments), S1
(untransformed), S2 (square root after adding the constant that brings the minimum to zero) and S3
(negative values dropped, the others square-rooted); L1 (yearly tree means) and L2 (campaign values)
for starch, sugar and Asat. A contrast is significant at adjusted p < 0.05. In the tables, `raw_diff`
is the difference of the raw means (first treatment minus second) of the rows the model uses (under S3,
without the dropped rows) and `estimate` the model contrast on the transformed scale. The water
potentials are not transformed, so S1, S2 and S3 give the same fit for them.

In [ ]:
LEAF_C = ["Starch", "Sugar", "Photo"]
GROWTH = ["Height_growth", "Diameter_growth", "AGB_growth"]
WP = ["Predawn", "Midday"]
YEARS = {"T1": [2019, 2020, 2021], "T2": [2020, 2021]}
SS = ("S1", "S2", "S3")
TT, fits = {}, []
t0 = time.time()
for sp in "BO":
    for trait in LEAF_C + GROWTH + WP:
        for T, yrs in YEARS.items():
            for L, data in (("L1", ty), ("L2", ca)):
                if L == "L2" and trait not in LEAF_C:
                    continue
                for S in SS:
                    r = o.treatment_test(data, sp, trait, yrs, trait not in WP, S)
                    TT[(sp, trait, T, S, L)] = r
                    fits.append({"species": sp, "trait": trait, "T": T, "L": L, "S": S, "n": r["n"],
                                 "transform": r["transform"], "converged": r["converged"]})
fits = pd.DataFrame(fits)
print(f"{len(fits)} fits in {time.time() - t0:.0f} s; not converged: {int((~fits.converged).sum())}")
print(fits.drop_duplicates(["species", "trait", "T", "L", "transform"]).to_string(index=False))
print("\nRows dropped under S3 (negative values), per species, trait, years and level:")
n12 = fits[fits.S != "S3"].groupby(["species", "trait", "T", "L"]).n.max()
n3 = fits[fits.S == "S3"].set_index(["species", "trait", "T", "L"]).n
print((n12 - n3).rename("dropped")[lambda s: s > 0].to_string())

def readings(trait):
    Ls = ("L1", "L2") if trait in LEAF_C else ("L1",)
    return [(T, S, L) for T in YEARS for S in SS for L in Ls]

def contrast(sp, trait, reading, year, label):
    T, S, L = reading
    c = TT[(sp, trait, T, S, L)]["contrasts"]
    return c[(c.year == year) & (c.contrast == label)].iloc[0]

def below(sp, trait, reading, year, label):
    r = contrast(sp, trait, reading, year, label)
    return bool(r.p_adj < 0.05 and r.estimate < 0)

def signif(sp, trait, reading, year, label):
    return bool(contrast(sp, trait, reading, year, label).p_adj < 0.05)

def table(sp, traits, label, years=(2020, 2021)):
    rows = []
    for trait in traits:
        for reading in readings(trait):
            for yr in years:
                r = contrast(sp, trait, reading, yr, label)
                rows.append({"species": sp, "trait": trait, "reading": "/".join(reading), "year": yr, "contrast": label,
                             "raw_diff": r.raw_diff, "estimate": r.estimate, "z": r.z, "p_adj": r.p_adj})
    return pd.DataFrame(rows)

### fig3-psi-warming-2020

> Warming made predawn leaf water potential more negative in 2020 in both species, and midday leaf water potential more negative in 2020 in beech only.

In [ ]:
print(pd.concat([table(sp, WP, "W-C", [2020]) for sp in "BO"]).round(4).to_string(index=False))
rows = []
for reading in readings("Predawn"):
    conds = {"PD beech": below("B", "Predawn", reading, 2020, "W-C"), "PD oak": below("O", "Predawn", reading, 2020, "W-C"),
             "MD beech": below("B", "Midday", reading, 2020, "W-C"), "MD oak not": not below("O", "Midday", reading, 2020, "W-C")}
    n = sum(conds.values())
    rows.append({"reading": "/".join(reading), **conds, "holding": n,
                 "outcome": "reproduced" if n == 4 else "partially_reproduced" if n >= 2 else "not_reproduced"})
print(pd.DataFrame(rows).to_string(index=False))
print("\nMeans of the tree-year means (MPa), 2020, monocultures:")
print(ty[(ty.Mixture == "Monoculture") & (ty.Year == 2020)].groupby(["Species", "treat"])[WP].mean().round(3).to_string())

**Result: partially reproduced.** In 2020, warming lowers predawn
potential in beech (W - C -0.67 MPa, adjusted p 0.0027 under T1, 0.027 under T2) and midday potential
in beech (-1.36 MPa, p 0.0005 and 0.0077), and midday potential in oak does not differ (p 0.38 and
0.55). Predawn potential in oak is lower under W by 0.23 MPa, but not significantly (p 0.39 and 0.55).
Three of the four conditions hold under both readings.

### fig3-psi-mr-oak-2020

> Lower soil moisture alone made predawn leaf water potential more negative in 2020 in oak only.

In [ ]:
print(pd.concat([table(sp, ["Predawn"], "MR-C", [2020]) for sp in "BO"]).round(4).to_string(index=False))
rows = []
for reading in readings("Predawn"):
    oak, beech = below("O", "Predawn", reading, 2020, "MR-C"), below("B", "Predawn", reading, 2020, "MR-C")
    rows.append({"reading": "/".join(reading), "oak MR<C": oak, "beech MR<C": beech,
                 "outcome": "reproduced" if oak and not beech else "not_reproduced"})
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** In 2020, predawn potential under MR is 0.28
MPa below C in oak but not significantly (adjusted p 0.25 under T1, 0.41 under T2); in beech it does
not differ either (p 0.91 and 0.97).

### fig3-psi-wmr-2021

> Combined warming and drier soil made predawn leaf water potential more negative in 2021 in both species.

In [ ]:
print(pd.concat([table(sp, ["Predawn"], "WMR-C", [2021]) for sp in "BO"]).round(4).to_string(index=False))
rows = []
for reading in readings("Predawn"):
    b, oo = below("B", "Predawn", reading, 2021, "WMR-C"), below("O", "Predawn", reading, 2021, "WMR-C")
    rows.append({"reading": "/".join(reading), "beech": b, "oak": oo,
                 "outcome": "reproduced" if b and oo else "partially_reproduced" if b or oo else "not_reproduced"})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** In 2021, predawn potential under WMR is below C in
beech (-0.61 MPa, adjusted p 0.0066 under T1, 0.039 under T2) and in oak (-0.72 MPa, p < 0.0001 under
both).

### disc-psi-minimum

> Under the low moisture treatment trees reached predawn water potentials of about -1.9 MPa in beech and -1.2 MPa in oak.

In [ ]:
m = ty[(ty.Mixture == "Monoculture") & ty.Year.isin([2020, 2021])]
mc = ca[(ca.Mixture == "Monoculture") & ca.Year.isin([2020, 2021])]
yearly = m.groupby(["Species", "treat", "Year"]).Predawn.mean()
camp = mc.groupby(["Species", "treat", "Year", "season"]).Predawn.mean()
rows = []
for sp, target, tol in (("B", -1.9, 0.19), ("O", -1.2, 0.12)):
    vals = {"a": yearly.loc[sp].loc[["MR"]].min(), "b": yearly.loc[sp].loc[["MR", "WMR"]].min(),
            "c": camp.loc[sp].loc[["MR", "WMR"]].min(), "d": mc[(mc.Species == sp) & mc.treat.isin(["MR", "WMR"])].Predawn.min()}
    for k, v in vals.items():
        rows.append({"species": sp, "reading": k, "value": v, "target": target, "within": within(v, target, tol)})
psi = pd.DataFrame(rows)
print(psi.round(3).to_string(index=False))
print("\nReadings with both species within tolerance:",
      [k for k in "abcd" if psi[psi.reading == k].within.all()])
print("\nYearly treatment means (MPa):"); print(yearly.unstack([1]).round(3).to_string())

**Result: not reproduced.** No reading puts both species within tolerance. The
oak value is within tolerance only under reading (c), the lowest campaign mean (-1.15 MPa against
-1.2); beech's is -0.68 (a), -1.06 (b), -1.18 (c) and -2.18 (d, one measurement) against -1.9 +-
0.19.

### fig4-beech-warming-none

> In beech, warming alone had no effect on starch, sugar, Asat, or height, diameter and AGWB increments.

In [ ]:
t = table("B", LEAF_C + GROWTH, "W-C")
print(t.round(4).to_string(index=False))
sig = t[t.p_adj < 0.05]
print("\nSignificant W-C contrasts:", len(sig), "of", len(t))
print(sig.round(4).to_string(index=False))
by = t.groupby("reading").apply(lambda g: int((g.p_adj < 0.05).sum())).rename("significant").reset_index()
print(by.to_string(index=False))
print("\nTable S1 warming terms (beech, sequential and marginal F), per reading:")
rows = []
for trait in LEAF_C + GROWTH:
    for reading in readings(trait):
        r = TT[("B", trait, *reading)]
        rows.append({"trait": trait, "reading": "/".join(reading),
                     "W_seq_F": r["anova"].F[0], "W_seq_p": r["anova"].p[0], "W_marg_F": r["anova_marginal"].F[0],
                     "W_marg_p": r["anova_marginal"].p[0], "WxY_marg_F": r["anova_marginal"].F[4], "WxY_marg_p": r["anova_marginal"].p[4]})
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Result: blocked (methods underspecified).** Of 108 W against C
contrasts in beech (six traits, two years, twelve readings), two are significant, both starch in 2020
under T1 on yearly means: under S2 (raw difference -3.52 %, adjusted p 0.041) and under S3 (p 0.018).
Under S1 the same contrast has p 0.099, and under T2 or on campaign values p 0.050 to 0.33. The answer
to the question named S1, S2 and S3 for the negative starch values; these readings still disagree, so
the claim stays blocked. The warming main effect is not significant in the marginal tests under T1
(p 0.081 to 0.72).

### fig4-oak-warming-growth-2021

> In oak, warming alone reduced height, diameter and AGWB increments, in 2021 only.

In [ ]:
t = table("O", GROWTH, "W-C")
print(t.round(4).to_string(index=False))
rows = []
for reading in readings("Height_growth"):
    n21 = sum(below("O", tr, reading, 2021, "W-C") for tr in GROWTH)
    n20 = sum(below("O", tr, reading, 2020, "W-C") for tr in GROWTH)
    out = ("reproduced" if n21 == 3 and n20 == 0 else "partially_reproduced" if n21 in (1, 2) and n20 == 0 else "not_reproduced")
    rows.append({"reading": "/".join(reading), "W<C in 2021": n21, "W<C in 2020": n20, "outcome": out})
print(pd.DataFrame(rows).to_string(index=False))

**Result: blocked (methods underspecified).** In oak in 2021, warming lowers
height increment (raw -11.7 cm; adjusted p 0.012 to 0.045 under S1 and S2, 0.052 and 0.054 under S3) and
AGWB increment (-21.5 g, p 0.032 under T1, 0.060 under T2), not diameter increment; nothing is
significant in 2020. Two or one of the three 2021 contrasts hold under five readings (partially
reproduced) and none under T2 with S3 (not reproduced). Reading S3, which the answer added for
negative increments, makes the readings differ.

### disc-oak-warming-no-carbon-change

> In oak, warming had no effect on Asat or on starch and sugar concentrations.

In [ ]:
t = table("O", LEAF_C, "W-C")
print(t.round(4).to_string(index=False))
print("\nSignificant W-C contrasts per reading:")
print(t.groupby("reading").apply(lambda g: int((g.p_adj < 0.05).sum())).to_string())

**Result: reproduced.** No W against C contrast of starch,
sugar or Asat in oak is significant in 2020 or 2021 under any of the twelve readings (T1 or T2, S1 to
S3, L1 or L2; smallest adjusted p 0.30).

### fig4-beech-mr

> In beech, lower soil moisture alone reduced Asat in 2021, height increment in 2021, diameter increment in 2020 and 2021, and AGWB increment in 2021.

In [ ]:
t = table("B", LEAF_C + GROWTH, "MR-C")
print(t.round(4).to_string(index=False))
named = [("Photo", 2021), ("Height_growth", 2021), ("Diameter_growth", 2020), ("Diameter_growth", 2021), ("AGB_growth", 2021)]
rows = []
for reading in [(T, S, L) for T in YEARS for S in SS for L in ("L1", "L2")]:
    hits = {f"{tr} {yr}": below("B", tr, reading if tr in LEAF_C else (reading[0], reading[1], "L1"), yr, "MR-C") for tr, yr in named}
    n = sum(hits.values())
    rows.append({"reading": "/".join(reading), **hits, "n": n,
                 "outcome": "reproduced" if n == 5 else "partially_reproduced" if n >= 3 else "not_reproduced"})
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** In beech, MR is below C for Asat in 2021 (raw
-5.90, adjusted p < 0.0001), height increment in 2021 (-9.6 cm, p 0.0004 to 0.0091) and AGWB increment in
2021 (-9.4 g, p 0.016 and 0.040) under every reading. Diameter increment in 2021 (-1.56) is below C
under S1 and S2 (p 0.008 to 0.025) but not under S3 (-0.35 on the rows kept, p 0.99), and diameter
increment in 2020 under none. Four of the five hold under S1 and S2 and three under S3, all partial.

### fig4-oak-mr

> In oak, lower soil moisture alone reduced Asat and height, diameter and AGWB increments in both 2020 and 2021.

In [ ]:
t = table("O", ["Photo"] + GROWTH, "MR-C")
print(t.round(4).to_string(index=False))

def count_table(sp, label, trait_years, full, partial_min):
    rows = []
    for reading in [(T, S, L) for T in YEARS for S in SS for L in ("L1", "L2")]:
        hits = {f"{tr} {yr}": below(sp, tr, reading if tr in LEAF_C else (reading[0], reading[1], "L1"), yr, label)
                for tr, yr in trait_years}
        n = sum(hits.values())
        rows.append({"reading": "/".join(reading), **hits, "n": n,
                     "outcome": "reproduced" if n == full else "partially_reproduced" if n >= partial_min else "not_reproduced"})
    return pd.DataFrame(rows)

print(count_table("O", "MR-C", [(tr, yr) for tr in ["Photo"] + GROWTH for yr in (2020, 2021)], 8, 4).to_string(index=False))

**Result: partially reproduced.** Six or seven of the eight trait-years hold under
every reading. Height increment in 2020 is not significant under any reading (raw -9.6 cm, smallest
adjusted p 0.062). Asat in 2021 is significant on campaign values under T1 with S1 or S2 and under T2
with S1 (p 0.039 to 0.048), not under S3 (p 0.15 and 0.19), and not on yearly means.

### fig4-beech-wmr

> In beech, combined warming and low moisture reduced Asat and diameter increment in 2020 and 2021.

In [ ]:
t = table("B", ["Photo", "Diameter_growth"], "WMR-C")
print(t.round(4).to_string(index=False))
print(count_table("B", "WMR-C", [(tr, yr) for tr in ["Photo", "Diameter_growth"] for yr in (2020, 2021)], 4, 2).to_string(index=False))

**Result: blocked (methods underspecified).** Asat under WMR is below C in 2021
(raw -6.40, adjusted p < 0.0001) but not in 2020 (p 0.062 to 0.23); diameter increment in 2021 (-1.19)
is significant under S2 (p 0.020 under T1, 0.045 under T2) but not under S1 (p 0.074 and 0.086) or S3
(-0.47 on the rows kept, p 0.81 and 0.80), and not in 2020. One of four holds under S1 and S3 (not
reproduced) and two under S2 (partially reproduced), so the readings differ.

### fig4-oak-wmr

> In oak, combined warming and low moisture reduced Asat and height, diameter and AGWB increments in 2020 and 2021.

In [ ]:
t = table("O", ["Photo"] + GROWTH, "WMR-C")
print(t.round(4).to_string(index=False))
print(count_table("O", "WMR-C", [(tr, yr) for tr in ["Photo"] + GROWTH for yr in (2020, 2021)], 8, 4).to_string(index=False))

**Result: reproduced.** In oak, WMR is below C for Asat, height, diameter and AGWB
increments in both 2020 and 2021, all eight under every reading, S3 included (adjusted p 0.0013 or less
for Asat; 0.028 to 0.048 for height in 2020; 0.029 or less for the others).

### fig4-wmr-same-as-mr

> Responses to combined warming and low moisture did not differ from responses to low moisture alone.

In [ ]:
t = pd.concat([table(sp, LEAF_C + GROWTH, "WMR-MR") for sp in "BO"])
sig = t[t.p_adj < 0.05]
print("Significant WMR-MR contrasts:", len(sig), "of", len(t))
print(sig.round(4).to_string(index=False))
print("\nPer reading (number significant):")
print(t.groupby("reading").apply(lambda g: int((g.p_adj < 0.05).sum())).to_string())
print("\nAll contrasts, absolute raw difference range per trait:")
print(t.groupby(["species", "trait"]).raw_diff.agg(lambda s: f"{s.abs().min():.3g} to {s.abs().max():.3g}").to_string())

**Result: blocked (methods underspecified).** Of 216 MR against WMR
contrasts, two are significant, both oak diameter increment in 2021 under S2 (raw WMR - MR -0.50,
adjusted p 0.020 under T1 and 0.033 under T2); under S1 and S3 none is. The readings differ.

### disc-nsc-unchanged-drier-soil

> Drier soils did not change starch and sugar concentrations in either species.

In [ ]:
t = pd.concat([table(sp, ["Starch", "Sugar"], "MR-C") for sp in "BO"])
print(t.round(4).to_string(index=False))
print("\nSignificant MR-C per reading:"); print(t.groupby("reading").apply(lambda g: int((g.p_adj < 0.05).sum())).to_string())
v = pd.concat([table(sp, ["Starch", "Sugar"], "WMR-C") for sp in "BO"])
print("\nVariant WMR-C, significant:", int((v.p_adj < 0.05).sum()), "of", len(v))
print(v[v.p_adj < 0.05].round(4).to_string(index=False))

**Result: reproduced.** No MR against C contrast of starch or
sugar is significant in either species or year under any of the twelve readings (smallest adjusted p
0.059, beech sugar in 2021 on campaign values under T2 and S3). In the WMR against C variant one of 96
is significant, beech starch in 2020 on campaign values under T1 and S3 (raw -3.80 %, p 0.028).

### disc-beech-asat-more-impaired

> Low soil moisture impaired Asat more strongly in beech than in oak.

In [ ]:
m = ty[(ty.Mixture == "Monoculture") & ty.Year.isin([2020, 2021])]
rows = []
for X in (["MR"], ["MR", "WMR"], ["WMR"]):
    for yrs in ([2020], [2021], [2020, 2021]):
        res = {}
        for sp in "BO":
            s = m[(m.Species == sp) & m.Year.isin(yrs)]
            c = s[s.treat == "C"].Photo.mean()
            x = s[s.treat.isin(X)].Photo.mean()
            res[sp] = (c, x)
        r1 = {sp: (c - x) / c for sp, (c, x) in res.items()}
        r2 = {sp: c - x for sp, (c, x) in res.items()}
        rows.append({"X": "+".join(X), "years": "+".join(map(str, yrs)), "C_beech": res["B"][0], "X_beech": res["B"][1],
                     "C_oak": res["O"][0], "X_oak": res["O"][1], "r1_beech": r1["B"], "r1_oak": r1["O"], "r2_beech": r2["B"],
                     "r2_oak": r2["O"], "r1_beech_larger": r1["B"] > r1["O"], "r2_beech_larger": r2["B"] > r2["O"]})
imp = pd.DataFrame(rows)
print(imp.round(3).to_string(index=False))
named = imp[imp.X != "WMR"]
print("\nReadings of the criterion (X = MR or MR+WMR): beech larger in",
      int(named.r1_beech_larger.sum() + named.r2_beech_larger.sum()), "of", 2 * len(named))

**Result: blocked (methods underspecified).** The reduction of
Asat is larger in beech in 6 of the 12 readings: in 2021 under both measures (MR: relative 0.72
against 0.30, absolute 5.9 against 3.4), and for the pooled years only as a relative reduction; in 2020
it is larger in oak (MR: 0.47 against 0.48, 1.6 against 5.0).

### tables6-no-seasonal-variation

> Asat, starch and sugar did not vary among the three campaigns of the growing season.

In [ ]:
S6Y = {"y1": [2019, 2020, 2021], "y2": [2020, 2021]}
rows = []
for trait in LEAF_C:
    for Y, yrs in S6Y.items():
        for trees in ("mono", "mono+mix"):
            base = ca[ca.Year.isin(yrs) & ((ca.Mixture == "Monoculture") if trees == "mono" else True)]
            for group in ("pooled", "B", "O"):
                sub0 = (base if group == "pooled" else base[base.Species == group]).dropna(subset=[trait])
                for S in SS:
                    sub = sub0.copy()
                    sub["y"], how = o.transform(sub[trait], True, S)
                    sub = sub.dropna(subset=["y"])
                    mdl = o.LME(sub, "y", "(C(W) + C(M) + C(season))**3")
                    for F, a in (("F1", mdl.anova_sequential()), ("F2", mdl.anova_marginal())):
                        rows.append({"trait": trait, "Y": Y, "trees": trees, "species": group, "S": S, "F": F,
                                     "transform": how, "n": len(sub), "converged": mdl.converged,
                                     "season_F": a.F[2], "season_p": a.p[2], "WS_F": a.F[4], "MS_F": a.F[5], "WMS_F": a.F[6]})
s6 = pd.DataFrame(rows)
print("Fits not converged:", int((~s6.converged).sum()))
print(s6.round(4).to_string(index=False))
by = s6.groupby(["Y", "trees", "species", "S", "F"]).season_p.apply(lambda p: bool((p > 0.05).all())).rename("all_three_p_over_005")
print("\nSeason term p > 0.05 for all three traits, per group:")
print(by.unstack("F").to_string())
rd = []
for Y in S6Y:
    for trees in ("mono", "mono+mix"):
        for S in SS:
            for F in ("F1", "F2"):
                a = bool(by.loc[(Y, trees, "pooled", S, F)])
                b = bool(by.loc[(Y, trees, "B", S, F)] and by.loc[(Y, trees, "O", S, F)])
                rd += [{"species": "a (pooled)", "Y": Y, "trees": trees, "S": S, "F": F, "absence_holds": a},
                       {"species": "b (each species)", "Y": Y, "trees": trees, "S": S, "F": F, "absence_holds": b}]
rd = pd.DataFrame(rd)
print("\nReadings of the criterion (a or b, y1 or y2, trees, S1 to S3, F1 or F2):")
print(rd.to_string(index=False))
print("Readings where the absence holds:", int(rd.absence_holds.sum()), "of", len(rd))
print(rd.groupby("F").absence_holds.agg(["sum", "count"]).to_string())
print("\nRange of the season term p per trait and kind of F test, over all readings:")
print(s6.groupby(["trait", "F"]).season_p.agg(["min", "max"]).round(4).to_string())
print("\nCampaign means (monocultures and mixtures, all years):")
print(ca.groupby(["season"])[LEAF_C].mean().round(3).to_string())
# Supplemental Table S6, F values of S, W x S, MR x S and W x MR x S
TABLE_S6 = {"Starch": [2.64, 0.05, 0.93, 0.24], "Sugar": [1.56, 0.04, 2.39, 0.64], "Photo": [2.03, 0.22, 0.79, 0.09]}
s6["F_paper"] = s6.trait.map(lambda t: TABLE_S6[t][0])
cmp_ = s6.assign(maxdiff=s6.apply(lambda r: max(abs(a - b) for a, b in zip([r.season_F, r.WS_F, r.MS_F, r.WMS_F], TABLE_S6[r.trait])), axis=1))
print("\nThe four season terms against Table S6, per trait and group: the 10 closest of", len(cmp_), "fits")
print(cmp_.sort_values("maxdiff").head(10)[["trait", "Y", "trees", "species", "S", "F", "season_F", "WS_F", "MS_F", "WMS_F", "maxdiff"]].round(3).to_string(index=False))
whole = cmp_.groupby(["Y", "trees", "species", "S", "F"]).maxdiff.max().sort_values()
print("\nLargest difference in F over all three traits and four terms, per reading (the 10 closest of", len(whole), "):")
print(whole.head(10).round(3).to_string())
print("Table S6:", TABLE_S6)

**Result: blocked (methods underspecified).** With sequential tests
(F1), at least one of starch, sugar and Asat has a season term with p <= 0.05 under all 24 readings
(Asat p <= 0.043 under every one). With marginal tests (F2), all three have p > 0.05 under 3 of the 24
readings, 2020 and 2021 (y2) on monocultures with both species pooled, under S1, S2 and S3 (starch p
0.12 to 0.26, sugar 0.17 to 0.32, Asat 0.070 to 0.12). The criterion now names F1 and F2 as readings,
and they differ; the first run, which read the tests as sequential, found the claim not reproduced.
Set against the paper's F values for the four season terms, the closest readings are y2 on
monocultures with both species pooled and marginal tests, under S2 and S1 (largest difference over
the 12 F values 0.99 and 1.01), the readings under which the absence holds; no reading matches
exactly, and with S3 the same reading differs by up to 3.1. This points to the reading but does not
settle it, so the claim stays blocked.

## Interaction tests and biodiversity effects (Supplemental Tables S2 and S3, Figure 5)

Interaction test of the claims file: per species, monoculture and mixture trees, square root,
lme(trait ~ (W + MR + year + I)^3, random = ~1|Chamber), and the mixture minus monoculture contrast
within each treatment and year (I1) or within each treatment averaged over 2020 and 2021 (I2), a
single contrast each (Wald z). The claims file does not say over which family these contrasts are
adjusted (asked in questions/dryad-4j0zpc8hz.yaml); both are run as readings: A1, each contrast alone
(unadjusted), and A2, single-step over the model's eight treatment and year contrasts (I1) or its four
treatment contrasts (I2), as the treatment test adjusts over the six pairs of a year. Years: Y1 (bud swelling and development duration on 2020 and 2021, the
other traits on 2019 to 2021), which the criteria use, and Y2 (2020 and 2021 for all traits), reported
as a variant. S1, S2 and S3 where values are negative, L1 and L2 for starch, sugar and Asat. In the
tables, `raw_mix_minus_mono` is the difference of the raw means of the rows the model uses (under S3,
without the dropped rows), and `raw_all` the same over all rows of the trait.

Biodiversity effects: NBE, complementarity (CE) and selection (SE) effects per mixture tree by the
claims file's Eqs. 2 to 4, against the mean of the monoculture trees of the species, treatment and
year (B1) or of the monoculture tree or trees of the species in the tree's own chamber (B2; every
chamber has one or two, and two are averaged), on yearly tree means (L1) or per campaign (L2), each
tested against zero by a one-sample t-test per species, treatment and year. With Eq. 3 as printed,
CE = 2 x NBE for every tree, so its t-test is the NBE's. Under B1 the reference is the same for every
tree of a group, so SE sums to zero and its mean is 0 (t = 0, p = 1) by construction. The legends'
Tukey's HSD against zero is not run: it is not defined for a comparison of one group with zero.

In [ ]:
PHENO = ["Flush_S1", "Dev_time", "Senes_10", "Senes_time", "Growth_length"]
# Years of the interaction model (claims file, opening comment): Y1, bud swelling and development duration on
# 2020 and 2021 and the other traits on 2019 to 2021, used by the criteria; Y2, 2020 and 2021 for all (variant).
IYEARS = {"Y1": lambda trait: [2020, 2021] if trait in ("Flush_S1", "Dev_time") else [2019, 2020, 2021],
          "Y2": lambda trait: [2020, 2021]}

# Adjustment of the mixture against monoculture contrasts, not stated in the claims file: A1 each contrast
# alone, A2 single-step over the model's eight treatment and year contrasts (I1) or four treatment contrasts (I2).
AA = ("A1", "A2")

def ireadings(trait):
    Ls = ("L1", "L2") if trait in LEAF_C else ("L1",)
    return [(Y, S, L, A) for Y in IYEARS for S in SS for L in Ls for A in AA]

IT = {}
t0 = time.time()
for sp in "BO":
    for trait in PHENO + LEAF_C + GROWTH:
        for Y, years_of in IYEARS.items():
            for L, data in (("L1", ty), ("L2", ca)):
                if L == "L2" and trait not in LEAF_C:
                    continue
                for S in SS:
                    r = o.interaction_test(data, sp, trait, years_of(trait), S)
                    IT[(sp, trait, Y, S, L)] = r
print(f"{len(IT)} interaction fits in {time.time() - t0:.0f} s; not converged:", sum(not r["converged"] for r in IT.values()))
print("Rows per fit, Y1 and Y2, S1 (S3 drops the negative values):")
print(pd.DataFrame([{"species": k[0], "trait": k[1], "Y": k[2], "S": k[3], "L": k[4], "n": r["n"], "transform": r["transform"]}
                    for k, r in IT.items() if k[3] in ("S1", "S3")]).pivot_table(index=["species", "trait", "L"], columns=["Y", "S"], values="n").to_string())

def itable(sp, traits, treat, years=("2020", "2021", "2020-21")):
    rows = []
    for trait in traits:
        for reading in ireadings(trait):
            fit = IT[(sp, trait, *reading[:3])]
            c = fit["contrasts"]
            for _, r in c[(c.treat == treat) & c.year.isin(years)].iterrows():
                rows.append({"species": sp, "trait": trait, "reading": "/".join(reading), "Y": reading[0], "A": reading[3], "treat": treat,
                             "I": "I2" if r.year == "2020-21" else "I1", "year": r.year, "transform": fit["transform"],
                             "raw_mix_minus_mono": r.raw_mix_minus_mono, "raw_all": r.raw_all, "estimate": r.estimate, "z": r.z,
                             "p": r.p if reading[3] == "A1" else r.p_family})
    return pd.DataFrame(rows)

BE = {}
for trait in PHENO + LEAF_C + GROWTH:
    for B in ("B1", "B2"):
        for L, data in (("L1", ty), ("L2", ca)):
            if L == "L2" and trait not in LEAF_C:
                continue
            eff = o.biodiversity_effects(data[data.Year.isin([2020, 2021])], trait, B, by_campaign=(L == "L2"))
            BE[(trait, B, L)] = o.t_tests(eff)

def absence_combos(sp, treat, nbe_years):
    """Per reading (Y, S, L, I, B): significant mixture-monoculture contrasts and NBEs in the years judged."""
    t = itable(sp, LEAF_C + GROWTH, treat)
    rows = []
    b_all = btable(sp, LEAF_C + GROWTH, treat, years=nbe_years, effects=("NBE",))
    for Y in IYEARS:
        for S in SS:
            for L in ("L1", "L2"):
                for A in AA:
                    for I in ("I1", "I2"):
                        g = t[(t.I == I) & t.apply(lambda r: r.reading == (f"{Y}/{S}/{L}/{A}" if r.trait in LEAF_C else f"{Y}/{S}/L1/{A}"), axis=1)]
                        nc = int((g.p < 0.05).sum())
                        for B in ("B1", "B2"):
                            b = b_all[(b_all.B == B) & ((b_all.L == L) | ~b_all.trait.isin(LEAF_C))]
                            nb_ = int((b.p < 0.05).sum())
                            rows.append({"Y": Y, "S": S, "L": L, "A": A, "I": I, "B": B, "contrasts_tested": len(g), "contrasts_significant": nc,
                                         "NBE_tested": len(b), "NBE_significant": nb_, "absence_holds": nc == 0 and nb_ == 0})
    return pd.DataFrame(rows)

def btable(sp, traits, treat, years=(2020, 2021), effects=("NBE", "CE", "SE")):
    rows = []
    for trait in traits:
        for B in ("B1", "B2"):
            for L in (("L1", "L2") if trait in LEAF_C else ("L1",)):
                t = BE[(trait, B, L)]
                t = t[(t.Species == sp) & (t.treat == treat) & t.Year.isin(years) & t.effect.isin(effects)]
                rows.append(t.assign(trait=trait, B=B, L=L))
    return pd.concat(rows)[["Species", "trait", "treat", "Year", "B", "L", "effect", "n", "mean", "t", "p"]]

### tables2-phenology-no-interaction

> Species mixture did not change phenology in either species, except a development duration about 2 days shorter in oak mixtures than monocultures under lower moisture, whose net biodiversity effect was not significant.

In [ ]:
t = pd.concat([itable(sp, PHENO, tr) for sp in "BO" for tr in o.TREATMENTS])
print("Phenology contrasts, mixture minus monoculture; significant ones:")
print(t[t.p < 0.05].round(4).to_string(index=False))
print("\nOak development duration under MR, every reading:")
print(t[(t.species == "O") & (t.trait == "Dev_time") & (t.treat == "MR")].round(4).to_string(index=False))
rows = []
for reading in ireadings("Dev_time"):
    for I in ("I1", "I2"):
        g = t[(t.reading == "/".join(reading)) & (t.I == I) & t.year.isin(["2020", "2021"] if I == "I1" else ["2020-21"])]
        s = g[g.p < 0.05]
        target = s[(s.species == "O") & (s.trait == "Dev_time") & (s.treat == "MR")]
        ok_target = bool(len(target) and (target.raw_mix_minus_mono < 0).all() and
                         target.raw_mix_minus_mono.map(lambda v: within(-v, 2, 0.2)).any())
        others = len(s) - len(target)
        rows.append({"reading": "/".join(reading), "Y": reading[0], "A": reading[3], "I": I, "significant": len(s), "oak Dev_time MR target holds": ok_target,
                     "other significant": others,
                     "contrast_outcome": "reproduced" if ok_target and others == 0 else "partially_reproduced" if ok_target and others == 1 else "not_reproduced"})
r2 = pd.DataFrame(rows)
print(r2.to_string(index=False))
print("\nContrast outcomes per Y and A (Y1 decides, Y2 variant):")
print(r2.groupby(["Y", "A", "I"]).agg(significant=("significant", lambda s: f"{s.min()} to {s.max()}"), outcomes=("contrast_outcome", lambda s: sorted(set(s)))).to_string())
print("\nNBE of oak development duration under MR (t-test against zero):")
print(btable("O", ["Dev_time"], "MR", effects=("NBE",)).round(4).to_string(index=False))

**Result: not reproduced.** Under Y1, the oak development duration
contrast under MR is not significant (raw mixture minus monoculture -0.83 days in 2020, -1.67 in 2021,
-1.25 over both years, p 0.14 to 0.67 each alone (A1), 0.63 to 1.0 adjusted over the model's
contrasts (A2)), and other phenology contrasts are: under A1, 6 per reading under I1 and 2 under I2,
and under A2, 2 under I1 and 2 under I2 (for example oak development duration under C in 2021, +5.5 days, p 0.0018, and oak onset of
senescence under W in 2020, +18.8 days, p 0.0008). The Y2 variant gives the same outcome under A1 and A2. The NBE of oak development duration under MR does not differ from zero (p 0.055 to
0.52), as the claim says.

### tables3-beech-warming-no-interaction

> Under warming, species mixture did not change beech leaf-level carbon relations or growth in 2020 and 2021, and beech showed no net biodiversity effect on these traits in 2021.

In [ ]:
t = itable("B", LEAF_C + GROWTH, "W")
print("Beech under W, mixture minus monoculture; significant contrasts:", int((t.p < 0.05).sum()), "of", len(t))
print(t[t.p < 0.05].round(4).to_string(index=False))
print("Significant per reading and I:"); print(t.groupby(["reading", "I"]).apply(lambda g: int((g.p < 0.05).sum())).to_string())
b = btable("B", LEAF_C + GROWTH, "W", years=(2020, 2021), effects=("NBE",))
print("\nNBE, beech under W:")
print(b.round(4).to_string(index=False))
print("2021 NBEs with p < 0.05:", int(((b.Year == 2021) & (b.p < 0.05)).sum()), "; 2020 (reported, not judged):", int(((b.Year == 2020) & (b.p < 0.05)).sum()))
combos = absence_combos("B", "W", (2021,))
print("\nPer reading (contrasts 2020 and 2021, NBE 2021):")
print(combos.to_string(index=False))
print("Readings where the absence holds:", int(combos.absence_holds.sum()), "of", len(combos))
print(combos.groupby(["Y", "A"]).absence_holds.agg(["sum", "count"]).to_string())

**Result: blocked (methods underspecified).** Under Y1 the absence
holds under 3 of 24 readings with each contrast alone (A1: I2 with B2 on yearly means, under S1, S2
and S3) and 6 of 24 with contrasts adjusted over the model's contrasts (A2: B2 on yearly means, I1 or
I2); under the others a mixture against monoculture contrast is significant (beech starch in 2020,
mixtures higher by 3.6 %, p 0.018 to 0.045 under I1 and A1, none under A2), or a 2021 NBE differs from zero (Asat, -60 to -67, p 0.0002 to 0.019, except
B2 on yearly means, p 0.078). The Y2 variant gives 2 (A1) and 6 (A2) of 24.

### figs2-beech-asat-selection-warming

> Under warming, beech Asat showed a positive selection effect, too weak to change the net biodiversity effect.

In [ ]:
b = btable("B", ["Photo"], "W")
print(b.round(4).to_string(index=False))
rows = []
for (B, L), g in b.groupby(["B", "L"]):
    hold = []
    for yr in (2020, 2021):
        se = g[(g.Year == yr) & (g.effect == "SE")].iloc[0]
        nbe = g[(g.Year == yr) & (g.effect == "NBE")].iloc[0]
        hold.append(bool(se["mean"] > 0 and se.p < 0.05 and not nbe.p < 0.05))
    rows.append({"B": B, "L": L, "holds_2020": hold[0], "holds_2021": hold[1], "reproduced": any(hold)})
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** Under no reading and in neither
year is the selection effect on beech Asat under W positive and significant: its mean is 0 under B1 on
yearly means (by construction), 11.6 and 3.3 (p 0.81 and 0.81) under B1 on campaigns, and -0.3 to -19
(p 0.32 to 1.0) under B2. Under B1 on yearly means the SE is 0 by construction with the
claims file's reading of Eq. 4, so that reading cannot show a selection effect; a question asks about it.

### tables3-oak-warming-diameter

> Under warming, oak in mixtures had a diameter increment about 0.3 cm per year lower than in monocultures, and no other difference between mixtures and monocultures.

In [ ]:
t = itable("O", LEAF_C + GROWTH, "W")
print("Oak under W, diameter increment:")
print(t[t.trait == "Diameter_growth"].round(4).to_string(index=False))
print("\nOther oak traits under W with p < 0.05:")
print(t[(t.trait != "Diameter_growth") & (t.p < 0.05)].round(4).to_string(index=False))
rows = []
for reading in ireadings("Diameter_growth"):
    for I in ("I1", "I2"):
        g = t[(t.reading == "/".join(reading)) & (t.I == I)]
        dia = g[g.trait == "Diameter_growth"]
        hits = dia[(dia.p < 0.05) & (dia.raw_mix_minus_mono < 0)]
        u1 = bool(hits.raw_mix_minus_mono.map(lambda v: within(-v, 0.3, 0.03)).any())
        u2 = bool(hits.raw_mix_minus_mono.map(lambda v: within(-v, 0.03, 0.003)).any())
        others = int((g[g.trait != "Diameter_growth"].p < 0.05).sum())
        rows.append({"reading": "/".join(reading), "Y": reading[0], "I": I, "diameter lower and significant": len(hits) > 0,
                     "within 0.3 (u1)": u1, "within 0.03 (u2, mm in cm)": u2, "other significant": others})
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** Under W, oak diameter increment in
mixtures is not significantly lower than in monocultures under any reading, with contrasts alone (A1)
or adjusted (A2) (raw mixture minus
monoculture -0.22 in 2020, p 0.20 to 0.45; +0.34 in 2021, +0.68 on the rows S3 keeps; +0.06 over both
years), and other oak contrasts are significant (AGWB increment, -8.9 g in 2020, p 0.012 under Y1 and A1; over both years -8.3 g, p 0.039 under Y1
and A2).

### fig5-oak-agwb-nbe-warming-2020

> Under warming in 2020, oak showed a negative net biodiversity effect on AGWB increment, explained by a negative complementarity effect.

In [ ]:
b = btable("O", ["AGB_growth"], "W", years=(2020, 2021))
print(b.round(4).to_string(index=False))
for B in ("B1", "B2"):
    g = b[(b.B == B) & (b.Year == 2020)].set_index("effect")
    print(B, "2020: NBE negative and p < 0.05:", bool(g.loc["NBE", "mean"] < 0 and g.loc["NBE", "p"] < 0.05),
          "; CE negative and p < 0.05:", bool(g.loc["CE", "mean"] < 0 and g.loc["CE", "p"] < 0.05))

**Result: blocked (methods underspecified).** For oak AGWB
increment under W in 2020, NBE and CE are negative and significant against the monoculture mean (B1,
NBE -446, p 0.0016) but not against the tree's own chamber (B2, -316, p 0.066). The two readings
differ.

### tables3-beech-mr-diameter

> Under lower soil moisture, beech in mixtures had a smaller diameter increment than in monocultures.

In [ ]:
t = itable("B", ["Diameter_growth"], "MR")
t["lower_and_significant"] = (t.p < 0.05) & (t.estimate < 0)
print(t.round(4).to_string(index=False))
print(t.groupby("reading").lower_and_significant.any().to_string())

**Result: not reproduced.** Under MR, beech diameter increment is
not lower in mixtures under any reading, A1 or A2: the 2020 difference is -0.20 (p 0.46 to 0.58 under
A1); in 2021 mixtures grow more under S1 and S2 (+0.67, p 0.018 to 0.082 under A1, 0.13 to 0.47 under
A2), and on the rows S3 keeps the difference is -0.17 (p 0.65 and 0.82 under A1); over both years it
is +0.22 (S1, S2, p 0.28 to 0.53 under A1) or -0.22 (S3, p 0.87 and 0.93 under A1).

### fig5-beech-sugar-nbe-mr

> Under low soil moisture, beech showed a negative net biodiversity effect on sugar concentration, driven by a negative complementarity effect.

In [ ]:
b = btable("B", ["Sugar"], "MR")
print(b.round(4).to_string(index=False))
rows = []
for (B, L), g in b.groupby(["B", "L"]):
    for yr in (2020, 2021):
        h = g[g.Year == yr].set_index("effect")
        rows.append({"B": B, "L": L, "year": yr,
                     "NBE<0 sig": bool(h.loc["NBE", "mean"] < 0 and h.loc["NBE", "p"] < 0.05),
                     "CE<0 sig": bool(h.loc["CE", "mean"] < 0 and h.loc["CE", "p"] < 0.05)})
r = pd.DataFrame(rows); r["both"] = r["NBE<0 sig"] & r["CE<0 sig"]
print(r.to_string(index=False))
print(r.groupby(["B", "L"]).both.any().rename("reproduced (2020 or 2021)").to_string())

**Result: reproduced.** In 2020, the NBE on beech sugar under MR is
negative and differs from zero under all four readings (-94 to -95, p 0.0025 to 0.036), and so does
the complementarity effect, which is twice the NBE (Eq. 3 as printed). In 2021 it is near zero (p >=
0.71). The year found is 2020, as the Discussion says; the sign of CE is negative, as the Results
say. With Eq. 3 read as in the claims file, CE is twice
the NBE for every tree, so the complementarity part is not tested apart from the NBE; a question asks
whether this is the paper's computation.

### oak-mr-no-interaction

> Under low soil moisture alone, oak showed no effect of species interactions and no net biodiversity effect.

In [ ]:
t = itable("O", LEAF_C + GROWTH, "MR")
print("Oak under MR, mixture minus monoculture; significant:", int((t.p < 0.05).sum()), "of", len(t))
print(t[t.p < 0.05].round(4).to_string(index=False))
print("Significant per reading and I:"); print(t.groupby(["reading", "I"]).apply(lambda g: int((g.p < 0.05).sum())).to_string())
b = btable("O", LEAF_C + GROWTH, "MR", effects=("NBE",))
print("\nNBE of oak under MR:"); print(b.round(4).to_string(index=False))
print("NBEs with p < 0.05:", int((b.p < 0.05).sum()), "of", len(b))
combos = absence_combos("O", "MR", (2020, 2021))
print("\nPer reading (contrasts and NBE, 2020 and 2021):")
print(combos.to_string(index=False))
print("Readings where the absence holds:", int(combos.absence_holds.sum()), "of", len(combos))
print(combos.groupby(["Y", "A"]).absence_holds.agg(["sum", "count"]).to_string())

**Result: blocked (methods underspecified).** Under Y1 the absence holds under 3
of 24 readings with contrasts alone (A1) and 3 of 24 with adjusted contrasts (A2), in both cases
contrasts per year (I1) with B2 on yearly means, under S1, S2 and S3. Over 2020 and
2021 (I2), oak Asat is higher in mixtures under MR (+2.18, p 0.0047 to 0.036 under every reading with A1, and
p 0.019 to 0.048 under A2 on yearly means or under S2 and S3 on campaigns), and
NBEs on Asat or sugar differ from zero under B1 or on campaign values (for example Asat 2021, +114,
p 0.031 under B1). The Y2 variant gives 3 (A1) and 6 (A2) of 24.

### tables3-beech-wmr-height

> Under combined warming and lower moisture, beech height increment was about 2 cm per year larger in monocultures than in mixtures.

In [ ]:
t = itable("B", ["Height_growth"], "WMR")
t["mono_minus_mix"] = -t.raw_mix_minus_mono
t["holds"] = (t.p < 0.05) & (t.estimate < 0) & t.mono_minus_mix.map(lambda v: within(v, 2, 0.2))
print(t.round(4).to_string(index=False))
print(t.groupby("reading").holds.any().to_string())

**Result: not reproduced.** Under WMR, beech height increment is 2.0 cm
larger in monocultures over both years (0.35 cm on the rows S3 keeps) and 0.9 and 3.1 cm in 2020 and
2021, but no contrast is significant under any reading (p 0.23 to 0.61 each alone, higher when adjusted).

### tables3-oak-wmr-diameter

> Under combined warming and lower moisture, oak diameter increment was 1.4 cm larger in mixtures than in monocultures.

In [ ]:
t = itable("O", ["Diameter_growth"], "WMR")
t["sig_larger"] = (t.p < 0.05) & (t.estimate > 0)
t["u1_within_1.4"] = t.raw_mix_minus_mono.map(lambda v: within(v, 1.4, 0.14))
t["u2_within_0.14"] = t.raw_mix_minus_mono.map(lambda v: within(v, 0.14, 0.014))
t["all_u1_within"] = t.raw_all.map(lambda v: within(v, 1.4, 0.14))
t["all_u2_within"] = t.raw_all.map(lambda v: within(v, 0.14, 0.014))
t["holds"] = t.sig_larger & (t["u1_within_1.4"] | t["u2_within_0.14"])
t["holds_on_all_rows"] = t.sig_larger & (t.all_u1_within | t.all_u2_within)
print(t.round(4).to_string(index=False))
print(t.groupby(["reading"])[["holds", "holds_on_all_rows"]].any().to_string())

**Result: blocked (methods underspecified).** Under WMR, oak diameter increment is
significantly larger in mixtures under every Y1 reading in 2021 when each contrast is taken alone (A1),
but the difference is within 1.4 +- 0.14 only under Y1 with S3, and only on the tree-years the S3
model keeps (+1.38, p 0.023). Over all trees the 2021 difference is +1.69 under every reading,
outside tolerance under both unit readings. Adjusted over the model's eight treatment and year
contrasts (A2), the same S3 contrast has p 0.16, while under S1 and S2 the 2021 contrast stays
significant (p 0.0017 to 0.0096) at +1.69. Under the Y2 variant with S3 the 2021 difference of 1.38
has p 0.051 (A1). The claim holds only for one choice of two details the claims file does not fix
(the family of the contrasts, and the rows over which the difference is taken under S3), so it is
blocked until the question on them is answered.

### fig5-oak-asat-nbe-wmr-2021

> Under combined warming and drier soils in 2021, oak showed a positive net biodiversity effect on Asat, explained by complementarity.

In [ ]:
b = btable("O", ["Photo"], "WMR", years=(2020, 2021))
print(b.round(4).to_string(index=False))
rows = []
for (B, L), g in b[b.Year == 2021].groupby(["B", "L"]):
    h = g.set_index("effect")
    nbe = bool(h.loc["NBE", "mean"] > 0 and h.loc["NBE", "p"] < 0.05)
    ce_sig = bool(h.loc["CE", "mean"] > 0 and h.loc["CE", "p"] < 0.05)
    out = "reproduced" if nbe and ce_sig else "partially_reproduced" if nbe and h.loc["CE", "mean"] > 0 else "not_reproduced"
    rows.append({"B": B, "L": L, "NBE>0 sig": nbe, "CE>0 sig": ce_sig, "outcome": out})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** In 2021 the NBE on oak Asat under WMR is
positive and differs from zero under all four readings (+193 to +194, p 0.0003 to 0.028), and so does
the complementarity effect (twice the NBE). With Eq. 3 read as in the claims file, CE is twice
the NBE for every tree, so the complementarity part is not tested apart from the NBE; a question asks
whether this is the paper's computation.

## Blocked claims: what the dataset holds

figs1-phenology-no-moisture, methods-treatment-magnitudes, fig6-soil-moisture-mr-equals-wmr,
tables5-annual-climate and tables4-initial-sizes-similar are blocked: the dataset has no soil moisture,
no chamber climate records and no initial sizes. The only air temperatures are the treatment means per
year and season printed below, from which part of two claims can be compared with the paper; these
comparisons decide no outcome.

In [ ]:
ann = d[d.Year.isin([2020, 2021])].groupby(["Year", "treat"]).Tannual.first().unstack()
print("Tannual per year and treatment (deg C), the dataset's only air temperature values:")
print(ann.round(2).to_string())
print("\nWarmed minus control, per year:")
print(pd.DataFrame({"W-C": ann.W - ann.C, "WMR-C": ann.WMR - ann.C}).round(2).to_string())
print("Mean of 2020 and 2021: W-C", round(float((ann.W - ann.C).mean()), 2), "; WMR-C", round(float((ann.WMR - ann.C).mean()), 2))
S5 = {(2020, "C"): 11.8, (2020, "MR"): 12.1, (2020, "W"): 17.0, (2020, "WMR"): 16.8,
      (2021, "C"): 10.5, (2021, "MR"): 10.8, (2021, "W"): 15.7, (2021, "WMR"): 15.6}
rows = [{"year": y, "treat": t, "dataset": ann.loc[y, t], "table_S5": v, "within_0.4": within(ann.loc[y, t], v, 0.4)} for (y, t), v in S5.items()]
print("\nAnnual air temperature against Table S5 (8 of the 16 values of tables5-annual-climate; soil moisture is not in the dataset):")
print(pd.DataFrame(rows).round(2).to_string(index=False))

## Summary

| Claim | Outcome |
|---|---|
| fig1ab-bud-swelling-advance | reproduced |
| fig1cd-development-duration | reproduced |
| fig1gh-senescence-duration | reproduced |
| fig1ij-growing-season | reproduced |
| fig1-oak-steeper-advance | reproduced |
| fig1ef-onset-senescence-no-temperature | reproduced |
| figs1-phenology-no-moisture | blocked (data missing) |
| fig2-gdd-chilling | reproduced |
| disc-max-chilling-days | reproduced |
| fig3-psi-warming-2020 | partially reproduced |
| fig3-psi-mr-oak-2020 | not reproduced |
| fig3-psi-wmr-2021 | reproduced |
| disc-psi-minimum | not reproduced |
| fig4-beech-warming-none | blocked (methods underspecified) |
| fig4-oak-warming-growth-2021 | blocked (methods underspecified) |
| disc-oak-warming-no-carbon-change | reproduced |
| fig4-beech-mr | partially reproduced |
| fig4-oak-mr | partially reproduced |
| fig4-beech-wmr | blocked (methods underspecified) |
| fig4-oak-wmr | reproduced |
| fig4-wmr-same-as-mr | blocked (methods underspecified) |
| disc-nsc-unchanged-drier-soil | reproduced |
| disc-beech-asat-more-impaired | blocked (methods underspecified) |
| tables6-no-seasonal-variation | blocked (methods underspecified) |
| tables2-phenology-no-interaction | not reproduced |
| tables3-beech-warming-no-interaction | blocked (methods underspecified) |
| figs2-beech-asat-selection-warming | not reproduced |
| tables3-oak-warming-diameter | not reproduced |
| fig5-oak-agwb-nbe-warming-2020 | blocked (methods underspecified) |
| tables3-beech-mr-diameter | not reproduced |
| fig5-beech-sugar-nbe-mr | reproduced |
| oak-mr-no-interaction | blocked (methods underspecified) |
| tables3-beech-wmr-height | not reproduced |
| tables3-oak-wmr-diameter | blocked (methods underspecified) |
| fig5-oak-asat-nbe-wmr-2021 | reproduced |
| methods-treatment-magnitudes | blocked (data missing) |
| fig6-soil-moisture-mr-equals-wmr | blocked (data missing) |
| tables5-annual-climate | blocked (data missing) |
| tables4-initial-sizes-similar | blocked (data missing) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's. The phenology regressions use the treatment's temperature
for each tree (the dataset has no chamber value) from an unnamed sensor. GDD and chilling days are the
deposited values. The mixed models are fitted in Python (statsmodels, REML), with nlme's tests and
multcomp-style single-step contrasts computed here; the paper used R. Exact duplicate rows are counted
once and one tree's two different rows for one campaign are averaged. Where the claims file leaves a
detail open, every reading is run, and a claim whose readings disagree is blocked. This version
follows the answers to the three questions of the first run: S3 (negative values dropped) is a
reading for starch, sugar, Asat and the increments, the interaction model uses the years of Y1 (Y2 as
a variant), and Table S6 is read under y1 and y2 with sequential and marginal F tests. After the
first verification round, the mixture against monoculture contrasts are run both alone (A1) and
adjusted over the model's contrasts (A2), as the claims file does not name the family; this, and the
rows over which the raw difference is taken under S3, decide tables3-oak-wmr-diameter, which is now
blocked. Two variants outside the claims file are shown: P1 with a random intercept per treatment and
year, under which fig1cd-development-duration (oak p 0.11) and fig1-oak-steeper-advance (p 0.097)
would not hold, and the Table S6 F values against the paper's. Three questions on these points are in
questions/dryad-4j0zpc8hz.yaml.

What was found about the dataset. The treatment codes in the data (C, D, H, HD) are not those of the
metadata (C, W, MR, WMR); the Warming and Soil moisture reduction columns give the mapping. 35 rows are
exact duplicates and one tree has two different rows for one campaign. Starch, sugar and Asat have
negative values. Growth_length is not onset of senescence minus bud swelling, as the Methods define
it, and neither the workbook nor Metadata.csv defines it. Diameter increments are labelled cm but are consistent with mm. Four compartments hold a
monoculture tree of one species and a mixture tree of the other. Soil moisture, the chamber climate
records and the initial sizes, on which five claims rest, are not deposited. Refitting the treatment
model reproduces Table S1 for some traits exactly, with marginal F tests.